# Flanker任务：VAM模型原理与一致性判断

本notebook演示：
1. 生成符合原始实验的Flanker刺激（目标鸟和干扰鸟颜色相同）
2. 解释VAM模型如何工作
3. 说明VAM如何通过漂移率推断一致性

## 1. 导入必要的库

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, classification_report
import os
import random

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# 设备配置
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

## 2. Flanker任务图像生成器（修正版）

In [ ]:
class FlankerStimulusGenerator:
    """
    Flanker任务刺激生成器
    
    重要：目标鸟和干扰鸟颜色完全相同，
    只有位置和方向不同，符合原始实验设计
    """
    
    def __init__(self, img_size=128, bird_size=20):
        self.img_size = img_size
        self.bird_size = bird_size
        self.directions = ['L', 'R', 'U', 'D']  # 左右上下
        self.direction_arrows = {
            'L': '◀',
            'R': '▶', 
            'U': '▲',
            'D': '▼'
        }
        
        # 布局定义：干扰项相对于目标的位置
        self.layouts = {
            'horizontal': [(-2, 0), (-1, 0), (1, 0), (2, 0)],
            'vertical': [(0, -2), (0, -1), (0, 1), (0, 2)],
            'cross': [(-1, 0), (1, 0), (0, -1), (0, 1)],
            'diagonal': [(-1, -1), (1, 1), (-1, 1), (1, -1)]
        }
        self.spacing = 25  # 鸟之间的间距
    
    def draw_bird(self, draw, center_x, center_y, direction):
        """
        绘制一个指向特定方向的鸟（用箭头表示）
        
        重要：目标鸟和干扰鸟使用相同的颜色（黑色）
        """
        arrow = self.direction_arrows[direction]
        
        # 使用更大的字体绘制箭头
        try:
            font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", self.bird_size)
        except:
            font = ImageFont.load_default()
        
        # 获取文本边界框
        bbox = draw.textbbox((0, 0), arrow, font=font)
        text_width = bbox[2] - bbox[0]
        text_height = bbox[3] - bbox[1]
        
        # 计算绘制位置（居中）
        x = center_x - text_width // 2
        y = center_y - text_height // 2
        
        # 绘制箭头（使用黑色，目标鸟和干扰鸟颜色相同）
        draw.text((x, y), arrow, font=font, fill='black')
    
    def generate_stimulus(self, target_dir, flanker_dir, layout='horizontal', 
                      target_pos=None, bg_color='white'):
        """
        生成单个Flanker刺激图像
        
        参数:
            target_dir: 目标方向 ('L', 'R', 'U', 'D')
            flanker_dir: 干扰项方向 ('L', 'R', 'U', 'D')
            layout: 布局类型
            target_pos: 目标位置 (x, y)，默认为图像中心
            bg_color: 背景颜色
            
        返回:
            PIL Image对象
        """
        # 创建空白图像
        img = Image.new('RGB', (self.img_size, self.img_size), bg_color)
        draw = ImageDraw.Draw(img)
        
        # 设置目标位置（默认为中心）
        if target_pos is None:
            target_pos = (self.img_size // 2, self.img_size // 2)
        
        # 绘制目标（中心，黑色）
        self.draw_bird(draw, target_pos[0], target_pos[1], target_dir)
        
        # 计算干扰项位置
        offsets = self.layouts[layout]
        for offset in offsets:
            flanker_pos = (
                target_pos[0] + offset[0] * self.spacing,
                target_pos[1] + offset[1] * self.spacing
            )
            # 绘制干扰项（周围，也是黑色）
            self.draw_bird(draw, flanker_pos[0], flanker_pos[1], flanker_dir)
        
        return img
    
    def generate_dataset(self, n_samples=1000):
        """
        生成完整的Flanker数据集
        
        返回:
            images: 图像数组 (n_samples, 3, img_size, img_size)
            labels: 一致性标签 (0=congruent, 1=incongruent)
            metadata: 元数据字典
        """
        images = []
        labels = []
        metadata = {
            'target_dirs': [],
            'flanker_dirs': [],
            'layouts': [],
            'positions': []
        }
        
        for i in range(n_samples):
            # 随机选择参数
            target_dir = random.choice(self.directions)
            flanker_dir = random.choice(self.directions)
            layout = random.choice(list(self.layouts.keys()))
            
            # 判断一致性
            is_congruent = (target_dir == flanker_dir)
            label = 0 if is_congruent else 1
            
            # 生成图像
            img = self.generate_stimulus(target_dir, flanker_dir, layout)
            
            # 转换为numpy数组并归一化
            img_array = np.array(img).astype(np.float32) / 255.0
            # 转换为CHW格式 (Channels, Height, Width)
            img_array = np.transpose(img_array, (2, 0, 1))
            
            images.append(img_array)
            labels.append(label)
            
            # 保存元数据
            metadata['target_dirs'].append(target_dir)
            metadata['flanker_dirs'].append(flanker_dir)
            metadata['layouts'].append(layout)
            metadata['positions'].append((self.img_size//2, self.img_size//2))
        
        return np.array(images), np.array(labels), metadata

## 3. 生成示例图像（修正：目标鸟和干扰鸟颜色相同）

In [ ]:
# 创建生成器
generator = FlankerStimulusGenerator(img_size=128, bird_size=24)

# 生成示例图像
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Flanker任务刺激示例（修正版：目标鸟和干扰鸟颜色相同）', fontsize=16)

# Congruent examples (目标=干扰)
congruent_examples = [
    ('L', 'L', 'horizontal'),
    ('R', 'R', 'vertical'),
    ('U', 'U', 'cross'),
    ('D', 'D', 'diagonal')
]

for i, (t_dir, f_dir, layout) in enumerate(congruent_examples):
    img = generator.generate_stimulus(t_dir, f_dir, layout)
    axes[0, i].imshow(img)
    axes[0, i].set_title(f'Congruent: 目标={t_dir}, 干扰={f_dir}\n布局={layout}', fontsize=10)
    axes[0, i].axis('off')

# Incongruent examples (目标≠干扰)
incongruent_examples = [
    ('L', 'R', 'horizontal'),
    ('R', 'L', 'vertical'),
    ('U', 'D', 'cross'),
    ('D', 'U', 'diagonal')
]

for i, (t_dir, f_dir, layout) in enumerate(incongruent_examples):
    img = generator.generate_stimulus(t_dir, f_dir, layout)
    axes[1, i].imshow(img)
    axes[1, i].set_title(f'Incongruent: 目标={t_dir}, 干扰={f_dir}\n布局={layout}', fontsize=10)
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

print("重要修正：")
print("- 所有箭头都是黑色（目标鸟和干扰鸟颜色相同）")
print("- 这符合原始Flanker任务实验设计")
print("- 被试需要通过位置（中心vs周围）来区分目标和干扰项")
print("- Congruent: 目标和干扰项方向相同")
print("- Incongruent: 目标和干扰项方向不同")

## 4. VAM模型架构与工作原理

In [ ]:
print("="*70)
print("VAM (Visual Attention Model) 模型架构")
print("="*70)
print()

print("【重要】VAM模型不直接输出'一致/不一致'的判断！")
print()

print("VAM模型的工作流程：")
print("─"*70)
print()
print("1. 输入：Flanker刺激图像")
print("   - 128×128×3 的彩色图像")
print("   - 包含5个箭头（1个目标+4个干扰）")
print("   - 所有箭头颜色相同（黑色）")
print()

print("2. CNN特征提取器")
print("   - 卷积层：提取视觉特征（边缘、纹理、形状）")
print("   - 池化层：压缩空间信息")
print("   - 全连接层：语义特征转换")
print()

print("3. 输出：4个漂移率")
print("   - 漂移率[0]：左方向的信息累积速度")
print("   - 漂移率[1]：右方向的信息累积速度")
print("   - 漂移率[2]：上方向的信息累积速度")
print("   - 漂移率[3]：下方向的信息累积速度")
print()

print("4. LBA认知模型")
print("   - 使用漂移率模拟决策过程")
print("   - 每个方向对应一个累加器")
print("   - 最快到达阈值的累加器获胜")
print("   - 生成预测的反应时间(RT)和选择")
print()

print("5. ELBO优化")
print("   - 比较预测RT/选择与真实数据")
print("   - 最大化证据下界")
print("   - 同时优化CNN和LBA参数")

## 5. VAM如何推断一致性？

In [ ]:
print("="*70)
print("VAM推断一致性的机制（间接推断）")
print("="*70)
print()

print("【关键】VAM通过分析漂移率分布来推断一致性")
print()

print("漂移率分布模式：")
print("─"*70)
print()

print("Congruent刺激（目标方向=干扰方向）：")
print("  示例：所有箭头都指向左")
print("  漂移率分布：[2.8, 0.3, 0.2, 0.1]")
print("              ↑    ↑    ↑    ↑")
print("              左   右   上   下")
print()
print("  特征：")
print("  - 目标方向(左)的漂移率最高：2.8")
print("  - 干扰方向(也是左)的漂移率也高")
print("  - 其他方向的漂移率很低")
print("  - 漂移率分布：一个峰值，非常集中")
print()

print("Incongruent刺激（目标方向≠干扰方向）：")
print("  示例：目标向左，干扰向右")
print("  漂移率分布：[2.1, 1.7, 0.3, 0.2]")
print("              ↑    ↑    ↑    ↑")
print("              左   右   上   下")
print()
print("  特征：")
print("  - 目标方向(左)的漂移率最高：2.1")
print("  - 干扰方向(右)的漂移率也较高：1.7")
print("  - 存在竞争：两个方向都有较高漂移率")
print("  - 漂移率分布：两个峰值，相对分散")

## 6. 一致性推断的具体方法

In [ ]:
def infer_congruence_from_drifts(drifts, threshold=0.7):
    """
    从漂移率推断一致性
    
    参数:
        drifts: 4个漂移率 [左, 右, 上, 下]
        threshold: 判断一致性的阈值
        
    返回:
        is_congruent: 是否一致
        confidence: 推断的置信度
    """
    # 找到最高和次高的漂移率
    sorted_drifts = sorted(drifts, reverse=True)
    max_drift = sorted_drifts[0]
    second_max_drift = sorted_drifts[1]
    
    # 计算集中度（最高漂移率相对于次高的比例）
    concentration = max_drift / (second_max_drift + 1e-6)
    
    # 判断一致性
    is_congruent = concentration > threshold
    
    # 计算置信度
    confidence = abs(concentration - threshold) / threshold
    
    return is_congruent, confidence

# 测试推断函数
print("="*70)
print("一致性推断示例")
print("="*70)
print()

# Congruent示例
congruent_drifts = [2.8, 0.3, 0.2, 0.1]
is_cong, conf = infer_congruence_from_drifts(congruent_drifts)
print(f"Congruent示例: {congruent_drifts}")
print(f"  推断: {'Congruent' if is_cong else 'Incongruent'}")
print(f"  置信度: {conf:.2f}")
print(f"  集中度: {congruent_drifts[0]/congruent_drifts[1]:.2f}")
print()

# Incongruent示例
incongruent_drifts = [2.1, 1.7, 0.3, 0.2]
is_cong, conf = infer_congruence_from_drifts(incongruent_drifts)
print(f"Incongruent示例: {incongruent_drifts}")
print(f"  推断: {'Congruent' if is_cong else 'Incongruent'}")
print(f"  置信度: {conf:.2f}")
print(f"  集中度: {incongruent_drifts[0]/incongruent_drifts[1]:.2f}")

## 7. 漂移率分布可视化

In [ ]:
# 可视化不同一致性条件下的漂移率分布
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

directions = ['左', '右', '上', '下']
x_pos = np.arange(len(directions))

# Congruent漂移率
congruent_drifts = [2.8, 0.3, 0.2, 0.1]
axes[0].bar(x_pos, congruent_drifts, color='green', alpha=0.7)
axes[0].set_xlabel('方向', fontsize=12)
axes[0].set_ylabel('漂移率', fontsize=12)
axes[0].set_title('Congruent刺激的漂移率分布\n(所有箭头向左)', fontsize=14)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(directions)
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].set_ylim([0, 3.0])

# Incongruent漂移率
incongruent_drifts = [2.1, 1.7, 0.3, 0.2]
axes[1].bar(x_pos, incongruent_drifts, color='orange', alpha=0.7)
axes[1].set_xlabel('方向', fontsize=12)
axes[1].set_ylabel('漂移率', fontsize=12)
axes[1].set_title('Incongruent刺激的漂移率分布\n(目标向左，干扰向右)', fontsize=14)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(directions)
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].set_ylim([0, 3.0])

plt.tight_layout()
plt.show()

print("关键观察：")
print("─"*70)
print("Congruent:")
print("  - 漂移率高度集中在一个方向")
print("  - 目标方向漂移率 >> 其他方向")
print("  - 反映视觉刺激的一致性")
print()
print("Incongruent:")
print("  - 漂移率分布在两个方向")
print("  - 目标和干扰方向都有较高漂移率")
print("  - 反映视觉刺激的冲突性")

## 8. VAM完整工作流程示例

In [ ]:
print("="*70)
print("VAM完整工作流程示例")
print("="*70)
print()

print("场景：目标向左，干扰向右（Incongruent）")
print("─"*70)
print()

print("步骤1：输入刺激")
print("  - 图像：128×128×3")
print("  - 内容：5个黑色箭头")
print("  - 中心：向左的箭头（目标）")
print("  - 周围：向右的箭头（干扰）")
print()

print("步骤2：CNN特征提取")
print("  - 卷积层检测到：")
print("    · 中心的左向箭头")
print("    · 周围的右向箭头")
print("    · 空间排列模式")
print("  - 特征编码：")
print("    · 位置信息（中心vs周围）")
print("    · 方向信息（左vs右）")
print("    · 冲突信息（方向不一致）")
print()

print("步骤3：生成漂移率")
print("  - CNN输出：[2.1, 1.7, 0.3, 0.2]")
print("  - 解释：")
print("    · 左方向(目标)漂移率最高：2.1")
print("    · 右方向(干扰)漂移率也高：1.7")
print("    · 反映了目标和干扰的竞争")
print()

print("步骤4：LBA模拟决策")
print("  - 4个累加器同时开始累积")
print("  - 左累加器：速度2.1")
print("  - 右累加器：速度1.7")
print("  - 上累加器：速度0.3")
print("  - 下累加器：速度0.2")
print("  - 结果：左累加器最快到达阈值")
print("  - 预测：选择左方向")
print()

print("步骤5：ELBO优化")
print("  - 比较预测与真实数据")
print("  - 如果被试确实选择左 → ELBO高")
print("  - 如果被试选择右 → ELBO低（模型需要调整）")
print("  - 反向传播更新CNN权重")
print()

print("步骤6：一致性推断（分析阶段）")
print("  - 分析漂移率分布：[2.1, 1.7, 0.3, 0.2]")
print("  - 计算集中度：2.1/1.7 = 1.24")
print("  - 与阈值比较：1.24 < 0.7")
print("  - 推断结果：Incongruent")
print("  - 置信度：高（两个方向竞争明显）")

## 9. 总结：VAM vs 直接分类器

In [ ]:
print("="*70)
print("VAM模型 vs 直接分类器：对比总结")
print("="*70)
print()

print("【直接分类器（如VGG+分类头）】：")
print("─"*70)
print("输入：Flanker刺激图像")
print("处理：CNN特征提取 → 全连接分类")
print("输出：Congruent / Incongruent（直接分类）")
print()
print("优点：")
  "- 简单直接，易于训练")
  "- 分类准确率高")
  "- 端到端优化")
print()
print("缺点：")
  "- 缺乏认知解释性")
  "- 无法预测反应时间")
  "- 不符合认知科学理论")
print()

print("【VAM模型】：")
print("─"*70)
print("输入：Flanker刺激图像")
print("处理：CNN特征提取 → 生成漂移率 → LBA认知模型")
print("输出：漂移率（间接推断一致性）")
print()
print("优点：")
  "- 具有认知科学理论支撑")
  "- 可以预测反应时间和选择")
  "- 漂移率有明确的认知意义")
  "- 可以分析注意力机制")
  "- 符合人类决策过程")
print()
print("缺点：")
  "- 模型复杂度高")
  "- 训练难度大（ELBO优化）")
  "- 一致性推断是间接的")
print()

print("【关键区别】：")
print("─"*70)
print("直接分类器：学习'视觉模式 → 一致性标签'的映射")
print("VAM模型：学习'视觉模式 → 漂移率 → 决策行为'的映射")
print()
print("VAM的优势在于：")
  "- 漂移率可以直接用于认知分析")
  "- 可以预测多种行为指标（RT、选择、错误率）")
  "- 提供了视觉-认知的桥梁")
  "- 一致性推断基于认知机制而非纯视觉模式")

## 10. 实际应用示例

In [ ]:
print("="*70)
print("VAM模型的实际应用")
print("="*70)
print()

print("应用1：注意力研究")
print("─"*70)
print("问题：被试如何处理目标和干扰项？")
print("VAM分析：")
  "- 比较Congruent vs Incongruent的漂移率")
  "- 量化注意力分配（目标vs干扰）")
  "- 分析干扰抑制效率")
  "- 研究个体差异")
print()

print("应用2：临床诊断")
print("─"*70)
print("问题：ADHD患者是否有注意力缺陷？")
print("VAM分析：")
  "- 比较患者vs正常人的漂移率模式")
  "- 检测干扰抑制能力差异")
  "- 量化注意力缺陷程度")
  "- 提供客观诊断指标")
print()

print("应用3：脑机接口")
print("─"*70)
print("问题：如何从脑信号解码注意力状态？")
print("VAM分析：")
  "- 建立脑活动-漂移率的映射")
  "- 实时解码注意力状态")
  "- 预测用户意图")
  "- 优化交互系统")
print()

print("应用4：AI系统设计")
print("─"*70)
print("问题：如何让AI更好地处理视觉冲突？")
print("VAM分析：")
  "- 学习人类的注意力机制")
  "- 改进AI的视觉处理")
  "- 提高鲁棒性")
  "- 增强可解释性")

## 11. 总结

In [ ]:
print("="*70)
print("总结：VAM模型与一致性判断")
print("="*70)
print()

print("✓ 修正：")
print("  - 目标鸟和干扰鸟颜色相同（黑色）")
print("  - 符合原始Flanker实验设计")
print("  - 被试通过位置区分目标和干扰")
print()

print("✓ VAM模型特点：")
print("  - 不直接输出一致性判断")
print("  - 输出4个方向的漂移率")
print("  - 通过漂移率分布间接推断一致性")
print()

print("✓ 一致性推断机制：")
print("  - Congruent：漂移率高度集中（一个峰值）")
print("  - Incongruent：漂移率相对分散（两个峰值）")
print("  - 使用集中度指标进行判断")
print()

print("✓ VAM的优势：")
print("  - 认知科学理论支撑")
print("  - 可预测多种行为指标")
print("  - 提供认知解释性")
print("  - 连接视觉与认知")
print()

print("✓ 关键洞察：")
print("  - VAM不是简单的分类器")
print("  - VAM是认知计算模型")
print("  - 漂移率是核心认知参数")
print("  - 一致性是漂移率分布的自然结果")